# AutoResearch-RL Demo Notebook

This notebook demonstrates how to run a single iteration of the
AutoResearch-RL research loop in a self-contained environment. It
loads the existing golden seed model, lets the meta-agent propose
a simple patch, and executes the resulting training run. The
demonstration is limited to a single iteration for brevity.


In [1]:
# Check installed versions of key libraries
import torch, numpy as np
print(f'Torch version: {torch.__version__}')
print(f'NumPy version: {np.__version__}')


Torch version: 2.10.0+cpu
NumPy version: 2.3.5


In [2]:
# Define a demonstration loop for a single iteration
import sys

# Ensure local package is discoverable
if '/home/oai/share' not in sys.path:
    sys.path.append('/home/oai/share')

from auto_research_rl.orchestrator.orchestrator import Orchestrator
from auto_research_rl.orchestrator.docker_runner import GPUDispatcher
from auto_research_rl.agent.ppo_agent import PPOMetaAgent
from auto_research_rl.agent.mdp_env import AutoResearchEnv
from auto_research_rl.gpu_cluster.sprt import SPRTFilter
from auto_research_rl.auditor.causality_auditor import check_causality_leak

def run_demo_loop():
    """Run a single iteration of the AutoResearch-RL loop."""
    orchestrator = Orchestrator()
    # The agent will fall back to a mock patch if no OPENAI_API_KEY is set
    agent = PPOMetaAgent(system_prompt='Minimize BPB under 16MB. No causality leaks. 10 min hard limit.')
    env = AutoResearchEnv(sota_bpb=1.0)
    # Load the golden seed model code
    with open('/home/oai/share/auto_research_rl/seed/train_gpt.py', 'r') as f:
        current_best_code = f.read()
    # Only one iteration for demonstration
    iteration = 1
    print(f'== Running iteration {iteration} ==')
    telemetry = {'last_vram_peak_mb': 78000, 'recent_oom': False, 'iteration': iteration}
    # Agent proposes a modification to the code
    candidate_code = agent.generate_action(current_best_code, env.history, telemetry)
    # Check causality compliance
    causality_leak = check_causality_leak(candidate_code)
    sprt = None
    # Perform smoke test and capacity check
    if causality_leak:
        result = orchestrator.submit_job(candidate_code)
        result.status = 'ABORTED'
        result.error_message = 'CausalityLeak'
    elif not orchestrator.run_smoke_test(candidate_code):
        result = orchestrator.submit_job(candidate_code)
    elif orchestrator.simulate_compression_and_capacity(candidate_code, num_parameters_int6=10000000, num_parameters_bf16=2000000) > 16000000:
        result = orchestrator.submit_job(candidate_code)
    else:
        sprt = SPRTFilter(sota_threshold=env.sota_bpb)
        def sprt_callback(step, loss):
            return sprt.update_and_check(step, loss)
        dispatcher = GPUDispatcher(sprt_callback=sprt_callback, time_limit_sec=60)
        job_id = f'job_iter_{iteration}'
        # Dispatch job for mock training; note that this uses the local
        # train_gpt.py script and simulates telemetry
        result = dispatcher.dispatch(job_id, candidate_code, num_parameters=12000000)
    # Calculate reward and update environment
    abort_step = 0
    step_info = env.step(result, action_patch='MOCK_PATCH_APPLIED', causality_leak=causality_leak, abort_step=abort_step)
    print('Job status:', result.status)
    print('Final BPB:', result.final_bpb)
    print('Reward:', step_info['reward'])
    # Accept candidate if it beats SOTA
    if result.status == 'COMPLETED' and result.final_bpb is not None and result.final_bpb < env.sota_bpb:
        print('New SOTA achieved!')
        current_best_code = candidate_code

2026-04-07 16:38:19,049 - Orchestrator - WARNING - zstandard module not found. Compression simulation will be approximate.


2026-04-07 16:38:19,051 - Orchestrator - WARNING - zstandard module not found. Compression simulation will be approximate.


In [3]:
# Run the demo loop
run_demo_loop()

2026-04-07 16:38:19,654 - Orchestrator - INFO - AutoResearch-RL Orchestrator Initialized.


2026-04-07 16:38:19,655 - PPO_Agent - INFO - No OpenAI API Key found. Operating in MOCK mode.


2026-04-07 16:38:19,656 - PPO_Agent - INFO - Querying Meta-Learner LLM for next action...


2026-04-07 16:38:19,656 - PPO_Agent - INFO - Successfully applied generated patch.


2026-04-07 16:38:19,666 - GPU_Dispatcher - INFO - Dispatching Job job_iter_1 to GPU Node...


== Running iteration 1 ==


2026-04-07 16:38:57,986 - MDP_Env - INFO - Reward Components: {'delta_bpb': -940.1560211181641, 'novelty_bonus': 0.1}


Job status: COMPLETED
Final BPB: 95.0156021118164
Reward: -940.056021118164


### Summary

This notebook executed a single iteration of the AutoResearch-RL loop
using a local CPU environment and the default mock patch provided
by the meta-agent. The telemetry and reward values above demonstrate
that the end-to-end pipeline runs successfully without external GPUs
or API keys. You can adjust the iteration count or integrate a custom
local LLM by modifying the `agent.generate_action` step.
